# Phase 1 — Encoder-Decoder with GQA

**Architecture**: Custom encoder + decoder with **Grouped-Query Attention** (GQA), `F.scaled_dot_product_attention` (Flash Attention), RoPE, SwiGLU, RMSNorm.

| Hyperparam | Value |
|---|---|
| d_model | 640 |
| encoder layers | 5 |
| decoder layers | 5 |
| query heads / KV heads | 10 / 5 (GQA, 2:1 grouping) |
| head_dim | 64 |
| d_ff (SwiGLU) | 1728 |
| enc / dec seq_len | 512 / 512 |
| **Total params** | **~135 M** |
| batch_size | 4 |
| grad_accum | 8 → effective 32 |
| optimizer | AdamW β=(0.9,0.95), wd=0.1 |
| lr | 3e-4 (warmup 500, cosine → 3e-5) |
| max_steps | 30 000 |
| mixed precision | AMP (fp16) |

**Key features**: GQA in both encoder and decoder ✅ | Flash-friendly SDPA ✅ | RoPE (self-attn only, not cross-attn) ✅ | SwiGLU ✅ | RMSNorm ✅ | KV cache for generation ✅ | Document masking ✅ | No EOS ✅


In [ ]:
!pip -q install tqdm huggingface_hub

import os, sys, json, math, time, random, pickle, warnings, shutil
from pathlib import Path
from dataclasses import dataclass, asdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

IS_KAGGLE = Path('/kaggle').exists()
print(f'ENV: {"Kaggle" if IS_KAGGLE else "Local"}')
print(f'torch={torch.__version__}  CUDA={torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name()}  ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)')


## 1 — Config

In [ ]:
@dataclass
class Config:
    d_model: int = 640
    enc_layers: int = 5
    dec_layers: int = 5
    n_heads: int = 10           # query heads
    n_kv_heads: int = 5         # KV heads → GQA with 2:1 grouping
    d_ff: int = 1728            # SwiGLU
    enc_seq_len: int = 512
    dec_seq_len: int = 512
    rope_theta: float = 10000.0
    rms_eps: float = 1e-5
    dropout: float = 0.0
    lr: float = 3e-4
    min_lr: float = 3e-5
    warmup_steps: int = 500
    max_steps: int = 30_000
    batch_size: int = 4
    grad_accum: int = 8
    weight_decay: float = 0.1
    max_grad_norm: float = 1.0
    betas: tuple = (0.9, 0.95)
    use_amp: bool = True
    log_interval: int = 50
    eval_interval: int = 500
    save_interval: int = 2500
    seed: int = 42
    gen_max_words: int = 128
    gen_temperature: float = 0.7
    gen_top_k: int = 50
    gen_top_p: float = 0.9
    gen_rep_penalty: float = 1.2
    def __post_init__(self):
        self.head_dim = self.d_model // self.n_heads
        assert self.d_model % self.n_heads == 0
        assert self.n_heads % self.n_kv_heads == 0

CFG = Config()
HF_REPO = 'Exploration-Lab/CS-781-Capstone'
TEST_FILE = 'Phase_1_Pretraining/Test/test.json'
EXP_NAME = 'encdec'

if IS_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        HF_TOKEN = ''
    OUT = Path('/kaggle/working'); CKPT = OUT/'encdec_ckpt'; CKPT.mkdir(exist_ok=True)
    DATA_DIR = Path('/kaggle/input')
else:
    HF_TOKEN = ''
    OUT = Path('.'); CKPT = OUT/'encdec_ckpt'; CKPT.mkdir(exist_ok=True)
    DATA_DIR = Path('./data')

random.seed(CFG.seed); np.random.seed(CFG.seed); torch.manual_seed(CFG.seed)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(CFG.seed)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Config: {CFG.d_model}d  enc={CFG.enc_layers}L  dec={CFG.dec_layers}L  q={CFG.n_heads} kv={CFG.n_kv_heads}  d_ff={CFG.d_ff}')
print(f'eff_batch={CFG.batch_size*CFG.grad_accum}  max_steps={CFG.max_steps}')


## 2 — Load tokenizer + data (no EOS, no meta.json)

In [ ]:
def _find(name, roots):
    for root in roots:
        root = Path(root)
        if not root.exists(): continue
        for p in [root/name] + list(root.rglob(name)):
            if p.exists(): return p
    raise FileNotFoundError(name)

roots = [DATA_DIR, Path('.'), Path('/kaggle/input'), Path('/kaggle/working')]
tok_py   = _find('a1_tokenizer.py', roots)
tok_json = _find('bpe_40k.json',    roots)
sys.path.insert(0, str(tok_py.parent))
from a1_tokenizer import BPETokenizer
tokenizer = BPETokenizer.load(str(tok_json))

# ── Vocab layout (no EOS) ──
VOCAB_SIZE   = tokenizer.vocab_size   # 40000
BOS_ID       = 2                      # <s> already in BPE specials
print(f'Tokenizer: BPE vocab={VOCAB_SIZE}  BOS_ID={BOS_ID} (<s>)')

# Load per-document arrays (no EOS in data)
train_pkl = _find('train_docs.pkl', roots)
val_pkl   = _find('val_docs.pkl',   roots)
with open(train_pkl, 'rb') as f: train_docs = pickle.load(f)
with open(val_pkl,   'rb') as f: val_docs   = pickle.load(f)
print(f'Documents: train={len(train_docs):,}  val={len(val_docs):,}')
print(f'Total tokens: train={sum(len(d) for d in train_docs)/1e9:.2f}B  val={sum(len(d) for d in val_docs)/1e6:.1f}M')


## 3 — Document-masked enc-dec dataset

In [ ]:
class DocMaskedEncDecDataset(Dataset):
    '''Each (enc, dec) pair is drawn from a single document → no cross-doc leakage.'''
    def __init__(self, docs, enc_len, dec_len):
        self.enc_len = enc_len; self.dec_len = dec_len
        # Need at least enc_len + dec_len tokens per doc
        self.docs = [d for d in docs if len(d) >= enc_len + dec_len]
        self.index = []
        for di, d in enumerate(self.docs):
            n = (len(d) - enc_len) // dec_len
            for wi in range(max(1, n)):
                self.index.append((di, wi))
    def __len__(self): return len(self.index)
    def __getitem__(self, i):
        di, wi = self.index[i]
        d = self.docs[di]
        max_start = len(d) - self.enc_len - self.dec_len
        start = (wi * self.dec_len) % max(1, max_start + 1)
        enc = d[start : start + self.enc_len]
        dec_full = d[start + self.enc_len : start + self.enc_len + self.dec_len]
        # Teacher forcing: dec_in = [BOS, t0, …, t_{L-2}] ; tgt = [t0, …, t_{L-1}]
        dec_in = np.concatenate([[BOS_ID], dec_full[:-1].astype(np.int64)]).astype(np.int64)
        tgt    = dec_full.astype(np.int64)
        return (torch.from_numpy(enc.astype(np.int64)),
                torch.from_numpy(dec_in),
                torch.from_numpy(tgt))

train_ds = DocMaskedEncDecDataset(train_docs, CFG.enc_seq_len, CFG.dec_seq_len)
val_ds   = DocMaskedEncDecDataset(val_docs,   CFG.enc_seq_len, CFG.dec_seq_len)
train_dl = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True,
                      num_workers=2, pin_memory=True, drop_last=True)
val_dl   = DataLoader(val_ds, batch_size=CFG.batch_size, shuffle=False,
                      num_workers=2, pin_memory=True, drop_last=True)
print(f'Train pairs: {len(train_ds):,}  Val pairs: {len(val_ds):,}')


## 4 — Model (GQA Encoder-Decoder with RoPE, SwiGLU, RMSNorm)

In [ ]:
# ── RMSNorm ──
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__(); self.eps = eps; self.g = nn.Parameter(torch.ones(d))
    def forward(self, x):
        return self.g * x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

# ── RoPE helpers ──
def build_rope_cache(seq_len, head_dim, theta=10000.0, device='cpu'):
    inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t = torch.arange(seq_len, device=device).float()
    freqs = torch.outer(t, inv_freq)
    return torch.cos(freqs), torch.sin(freqs)

def apply_rope(x, cos, sin):
    T = x.size(-2)
    cos = cos[:T].unsqueeze(0).unsqueeze(0)
    sin = sin[:T].unsqueeze(0).unsqueeze(0)
    x1 = x[..., 0::2]; x2 = x[..., 1::2]
    r1 = x1 * cos - x2 * sin
    r2 = x1 * sin + x2 * cos
    return torch.stack([r1, r2], dim=-1).flatten(-2)

# ── GQA Self-Attention (with RoPE, KV cache, configurable causal) ──
class GQASelfAttention(nn.Module):
    def __init__(self, cfg, is_causal=True):
        super().__init__()
        self.n_q = cfg.n_heads; self.n_kv = cfg.n_kv_heads
        self.head_dim = cfg.head_dim; self.n_rep = self.n_q // self.n_kv
        self.is_causal = is_causal
        self.wq = nn.Linear(cfg.d_model, self.n_q*self.head_dim, bias=False)
        self.wk = nn.Linear(cfg.d_model, self.n_kv*self.head_dim, bias=False)
        self.wv = nn.Linear(cfg.d_model, self.n_kv*self.head_dim, bias=False)
        self.wo = nn.Linear(self.n_q*self.head_dim, cfg.d_model, bias=False)
    def forward(self, x, cos, sin, kv_cache=None, start_pos=0):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.n_q,  self.head_dim).transpose(1, 2)
        k = self.wk(x).view(B, T, self.n_kv, self.head_dim).transpose(1, 2)
        v = self.wv(x).view(B, T, self.n_kv, self.head_dim).transpose(1, 2)
        cos_s = cos[start_pos : start_pos+T]
        sin_s = sin[start_pos : start_pos+T]
        q = apply_rope(q, cos_s, sin_s)
        k = apply_rope(k, cos_s, sin_s)
        if kv_cache is not None:
            pk, pv = kv_cache
            k = torch.cat([pk, k], dim=2); v = torch.cat([pv, v], dim=2)
        new_cache = (k, v)
        if self.n_rep > 1:
            k_e = k.repeat_interleave(self.n_rep, dim=1)
            v_e = v.repeat_interleave(self.n_rep, dim=1)
        else:
            k_e = k; v_e = v
        causal = self.is_causal and (kv_cache is None)
        out = F.scaled_dot_product_attention(q, k_e, v_e, is_causal=causal)
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.wo(out), new_cache

# ── GQA Cross-Attention (no RoPE, KV from encoder memory, cache once) ──
class GQACrossAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_q = cfg.n_heads; self.n_kv = cfg.n_kv_heads
        self.head_dim = cfg.head_dim; self.n_rep = self.n_q // self.n_kv
        self.wq = nn.Linear(cfg.d_model, self.n_q*self.head_dim, bias=False)
        self.wk = nn.Linear(cfg.d_model, self.n_kv*self.head_dim, bias=False)
        self.wv = nn.Linear(cfg.d_model, self.n_kv*self.head_dim, bias=False)
        self.wo = nn.Linear(self.n_q*self.head_dim, cfg.d_model, bias=False)
    def forward(self, x, memory, kv_cache=None):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.n_q, self.head_dim).transpose(1, 2)
        if kv_cache is not None:
            k, v = kv_cache
        else:
            S = memory.size(1)
            k = self.wk(memory).view(B, S, self.n_kv, self.head_dim).transpose(1, 2)
            v = self.wv(memory).view(B, S, self.n_kv, self.head_dim).transpose(1, 2)
        new_cache = (k, v)
        if self.n_rep > 1:
            k_e = k.repeat_interleave(self.n_rep, dim=1)
            v_e = v.repeat_interleave(self.n_rep, dim=1)
        else:
            k_e = k; v_e = v
        out = F.scaled_dot_product_attention(q, k_e, v_e, is_causal=False)
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.wo(out), new_cache

# ── SwiGLU FFN ──
class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)
        self.w2 = nn.Linear(d_ff, d_model, bias=False)
        self.w3 = nn.Linear(d_model, d_ff, bias=False)
    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

# ── Encoder Block ──
class EncoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n1 = RMSNorm(cfg.d_model, cfg.rms_eps)
        self.attn = GQASelfAttention(cfg, is_causal=False)   # bidirectional
        self.n2 = RMSNorm(cfg.d_model, cfg.rms_eps)
        self.ff = SwiGLU(cfg.d_model, cfg.d_ff)
    def forward(self, x, cos, sin):
        a, _ = self.attn(self.n1(x), cos, sin)
        x = x + a
        x = x + self.ff(self.n2(x))
        return x

# ── Decoder Block ──
class DecoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n1 = RMSNorm(cfg.d_model, cfg.rms_eps)
        self.self_attn = GQASelfAttention(cfg, is_causal=True)   # causal
        self.n2 = RMSNorm(cfg.d_model, cfg.rms_eps)
        self.cross_attn = GQACrossAttention(cfg)
        self.n3 = RMSNorm(cfg.d_model, cfg.rms_eps)
        self.ff = SwiGLU(cfg.d_model, cfg.d_ff)
    def forward(self, x, memory, cos, sin, self_kv=None, cross_kv=None, start_pos=0):
        a, new_self_kv = self.self_attn(self.n1(x), cos, sin, self_kv, start_pos)
        x = x + a
        c, new_cross_kv = self.cross_attn(self.n2(x), memory, cross_kv)
        x = x + c
        x = x + self.ff(self.n3(x))
        return x, new_self_kv, new_cross_kv

# ── Encoder-Decoder LM ──
class EncDecGQA(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.src_emb = nn.Embedding(VOCAB_SIZE, cfg.d_model)
        self.tgt_emb = self.src_emb  # weight tying — same vocab, shared embedding
        self.enc_blocks = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.enc_layers)])
        self.dec_blocks = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.dec_layers)])
        self.enc_norm = RMSNorm(cfg.d_model, cfg.rms_eps)
        self.dec_norm = RMSNorm(cfg.d_model, cfg.rms_eps)
        self.head = nn.Linear(cfg.d_model, VOCAB_SIZE, bias=False)
        self.head.weight = self.src_emb.weight  # tie output head to embedding
        max_len = max(cfg.enc_seq_len, cfg.dec_seq_len) * 4
        cos, sin = build_rope_cache(max_len, cfg.head_dim, cfg.rope_theta)
        self.register_buffer('rope_cos', cos, persistent=False)
        self.register_buffer('rope_sin', sin, persistent=False)
        self.apply(self._init)
    def _init(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def encode(self, src):
        x = self.src_emb(src)
        for blk in self.enc_blocks:
            x = blk(x, self.rope_cos, self.rope_sin)
        return self.enc_norm(x)

    def forward(self, src, tgt_in, targets=None):
        memory = self.encode(src)
        x = self.tgt_emb(tgt_in)
        for blk in self.dec_blocks:
            x, _, _ = blk(x, memory, self.rope_cos, self.rope_sin)
        logits = self.head(self.dec_norm(x))
        if targets is None: return logits, None
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

    def decode_cached(self, tgt_in, memory, self_caches, cross_caches, start_pos):
        '''Single-token decoding with KV caches for fast generation.'''
        x = self.tgt_emb(tgt_in)
        new_self, new_cross = [], []
        for i, blk in enumerate(self.dec_blocks):
            sc = self_caches[i] if self_caches else None
            cc = cross_caches[i] if cross_caches else None
            x, nsc, ncc = blk(x, memory, self.rope_cos, self.rope_sin, sc, cc, start_pos)
            new_self.append(nsc); new_cross.append(ncc)
        logits = self.head(self.dec_norm(x))
        return logits, new_self, new_cross

model = EncDecGQA(CFG).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model params: {n_params/1e6:.1f} M')

# ── GPU VRAM smoke test ──
if DEVICE == 'cuda':
    torch.cuda.reset_peak_memory_stats()
    dummy_src = torch.randint(0, VOCAB_SIZE, (CFG.batch_size, CFG.enc_seq_len), device=DEVICE)
    dummy_din = torch.randint(0, VOCAB_SIZE, (CFG.batch_size, CFG.dec_seq_len), device=DEVICE)
    dummy_tgt = torch.randint(0, VOCAB_SIZE, (CFG.batch_size, CFG.dec_seq_len), device=DEVICE)
    with torch.cuda.amp.autocast(enabled=CFG.use_amp):
        _, loss = model(dummy_src, dummy_din, targets=dummy_tgt)
    loss.backward()
    peak = torch.cuda.max_memory_allocated() / 1e9
    model.zero_grad(set_to_none=True)
    del dummy_src, dummy_din, dummy_tgt, loss
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'\n🔥 VRAM smoke test (1 batch, fwd+bwd):')
    print(f'   Peak VRAM used : {peak:.2f} GB')
    print(f'   GPU total      : {total_gb:.1f} GB')
    print(f'   Headroom       : {total_gb - peak:.2f} GB')
    tok_seen = CFG.max_steps * CFG.batch_size * CFG.grad_accum * CFG.dec_seq_len
    print(f'   Tokens seen    : {CFG.max_steps} steps × {CFG.batch_size*CFG.grad_accum} eff × {CFG.dec_seq_len} dec_len = {tok_seen/1e6:.0f}M')
    if peak > total_gb * 0.95:
        print(f'   ⚠️  WARNING: Peak VRAM near limit — may OOM during training!')
    else:
        print(f'   ✅ Looks good — should fit in GPU memory')


## 5 — Optimizer + training loop

In [ ]:
def get_lr(step):
    if step < CFG.warmup_steps: return CFG.lr * (step+1)/CFG.warmup_steps
    if step >= CFG.max_steps: return CFG.min_lr
    prog = (step - CFG.warmup_steps) / max(1, CFG.max_steps - CFG.warmup_steps)
    return CFG.min_lr + (CFG.lr - CFG.min_lr) * 0.5 * (1 + math.cos(math.pi * prog))

decay, no_decay = [], []
for n, p in model.named_parameters():
    (no_decay if p.ndim < 2 else decay).append(p)
opt = torch.optim.AdamW([
    {'params': decay, 'weight_decay': CFG.weight_decay},
    {'params': no_decay, 'weight_decay': 0.0},
], lr=CFG.lr, betas=CFG.betas)
scaler = torch.cuda.amp.GradScaler(enabled=CFG.use_amp)

@torch.no_grad()
def evaluate(max_batches=50):
    model.eval(); tot = n = 0
    for i, (s, di, y) in enumerate(val_dl):
        if i >= max_batches: break
        s, di, y = s.to(DEVICE), di.to(DEVICE), y.to(DEVICE)
        with torch.cuda.amp.autocast(enabled=CFG.use_amp):
            _, loss = model(s, di, targets=y)
        tot += loss.item(); n += 1
    model.train()
    return tot/max(n,1)

# Resume
start_step = 0; best_val = float('inf')
history = {'train_loss': [], 'val_loss': [], 'lr': []}
ckpts = sorted(CKPT.glob('step_*.pt'), key=lambda p: int(p.stem.split('_')[1]))
if (CKPT/'final.pt').exists(): resume = CKPT/'final.pt'
elif ckpts: resume = ckpts[-1]
elif (CKPT/'best.pt').exists(): resume = CKPT/'best.pt'
else: resume = None
if resume:
    ck = torch.load(resume, map_location=DEVICE)
    model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
    scaler.load_state_dict(ck['scaler']); start_step = ck.get('step', 0)
    best_val = ck.get('best_val', best_val); history = ck.get('history', history)
    print(f'Resumed from {resume.name} @ step {start_step}')
else:
    print('Starting from scratch')


In [ ]:
model.train()
step = start_step
pbar = tqdm(total=CFG.max_steps, initial=step, desc='train')
t0 = time.time()
train_iter = iter(train_dl)
accum_loss = 0.0

while step < CFG.max_steps:
    lr = get_lr(step)
    for g in opt.param_groups: g['lr'] = lr
    opt.zero_grad(set_to_none=True)

    for _ in range(CFG.grad_accum):
        try: s, di, y = next(train_iter)
        except StopIteration:
            train_iter = iter(train_dl); s, di, y = next(train_iter)
        s, di, y = s.to(DEVICE), di.to(DEVICE), y.to(DEVICE)
        with torch.cuda.amp.autocast(enabled=CFG.use_amp):
            _, loss = model(s, di, targets=y)
            loss = loss / CFG.grad_accum
        scaler.scale(loss).backward()
        accum_loss += loss.item()

    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
    scaler.step(opt); scaler.update()
    step += 1
    pbar.update(1); pbar.set_postfix(loss=f'{accum_loss:.3f}', lr=f'{lr:.2e}')

    if step % CFG.log_interval == 0:
        ppl = math.exp(min(accum_loss, 20))
        history['train_loss'].append(accum_loss); history['lr'].append(lr)
        print(f'  step {step:>6d}/{CFG.max_steps}  loss={accum_loss:.4f}  ppl={ppl:.1f}  lr={lr:.2e}')
    accum_loss = 0.0

    if step % CFG.eval_interval == 0 or step == CFG.max_steps:
        vl = evaluate()
        vl_ppl = math.exp(min(vl, 20))
        history['val_loss'].append((step, vl))
        print(f'  ✦ [step {step}] val_loss={vl:.4f}  val_ppl={vl_ppl:.1f}  best={best_val:.4f}')
        if vl < best_val:
            best_val = vl
            torch.save({'model': model.state_dict(), 'opt': opt.state_dict(),
                        'scaler': scaler.state_dict(), 'step': step,
                        'best_val': best_val, 'history': history,
                        'cfg': asdict(CFG)}, CKPT/'best.pt')
            print(f'  ✓ new best → best.pt')

    if step % CFG.save_interval == 0:
        torch.save({'model': model.state_dict(), 'opt': opt.state_dict(),
                    'scaler': scaler.state_dict(), 'step': step,
                    'best_val': best_val, 'history': history,
                    'cfg': asdict(CFG)}, CKPT/f'step_{step}.pt')

pbar.close()
torch.save({'model': model.state_dict(), 'opt': opt.state_dict(),
            'scaler': scaler.state_dict(), 'step': step,
            'best_val': best_val, 'history': history,
            'cfg': asdict(CFG)}, CKPT/'final.pt')
print(f'\n✓ Training done in {(time.time()-t0)/60:.1f} min')


## 6 — Generation with KV cache (128-word continuation)

In [ ]:
ck = torch.load(CKPT/'best.pt', map_location=DEVICE)
model.load_state_dict(ck['model']); model.eval()
print(f'Loaded best (step {ck["step"]}, val_loss={ck["best_val"]:.4f})')

@torch.no_grad()
def generate_128_words(prompt, max_new=800):
    src_ids = tokenizer.encode(prompt)[:CFG.enc_seq_len]
    src = torch.tensor([src_ids], dtype=torch.long, device=DEVICE)
    with torch.cuda.amp.autocast(enabled=CFG.use_amp):
        memory = model.encode(src)

    # Prefill with BOS
    xi = torch.tensor([[BOS_ID]], dtype=torch.long, device=DEVICE)
    with torch.cuda.amp.autocast(enabled=CFG.use_amp):
        logits, self_caches, cross_caches = model.decode_cached(
            xi, memory, None, None, start_pos=0)
    pos = 1
    out_ids = []
    recent = []

    for _ in range(max_new):
        last = logits[0, -1].float()
        if recent:
            last[recent] /= CFG.gen_rep_penalty
        last = last / CFG.gen_temperature
        topk_v, topk_i = torch.topk(last, CFG.gen_top_k)
        probs = F.softmax(topk_v, dim=-1)
        sorted_p, sorted_idx = torch.sort(probs, descending=True)
        cum = torch.cumsum(sorted_p, dim=-1)
        cutoff = cum > CFG.gen_top_p; cutoff[0] = False
        sorted_p[cutoff] = 0.0
        sorted_p = sorted_p / sorted_p.sum()
        pick = torch.multinomial(sorted_p, 1).item()
        tok = topk_i[sorted_idx[pick]].item()
        out_ids.append(tok)
        recent.append(tok); recent = recent[-50:]
        decoded = tokenizer.decode(out_ids)
        if len(decoded.split()) >= CFG.gen_max_words: break
        # Incremental decode
        xi = torch.tensor([[tok]], dtype=torch.long, device=DEVICE)
        with torch.cuda.amp.autocast(enabled=CFG.use_amp):
            logits, self_caches, cross_caches = model.decode_cached(
                xi, memory, self_caches, cross_caches, start_pos=pos)
        pos += 1

    text = tokenizer.decode(out_ids)
    words = text.split()[:CFG.gen_max_words]
    while len(words) < CFG.gen_max_words:
        words.append('.')
    return ' '.join(words)

sample = 'The Supreme Court of India observed that'
cont = generate_128_words(sample)
print(f'Prompt: {sample}')
print(f'Continuation ({len(cont.split())} words):\n{cont}')


## 7 — Produce submission on test.json

In [ ]:
from huggingface_hub import hf_hub_download
test_path = hf_hub_download(HF_REPO, TEST_FILE, repo_type='dataset', token=HF_TOKEN)
with open(test_path) as f: test_data = json.load(f)
print(f'Test prompts: {len(test_data)}')

results = {}
for i, (tid, item) in enumerate(tqdm(test_data.items(), desc='generating')):
    prompt = item['text'] if isinstance(item, dict) else item
    cont = generate_128_words(prompt)
    results[tid] = {'text': cont}

with open(OUT/'submission_encdec.json', 'w') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(f'✓ Wrote {OUT}/submission_encdec.json  ({len(results)} items)')
